# Capítulo 7 — Desempenho de Arquiteturas Cliente-Servidor

**Referência:** Capítulo 7 do livro-texto  
**Biblioteca:** `line_solver` — `pfqn_mvald` (MVA exato para redes com taxas dependentes de carga)


---

#### Consider the CS model discussed in the telemarketing company probelm and assume that the server will be upgraded by a two processor server where each processor is identical to the original processor. If the same disk is used, solve the CS model assuming that the service rate, $\mu(j)$, for the server CPU is such that $\mu(2) = 1.8\mu(1)$.

SQL requested related parameters

- $N_{sql}$ = 4 SQL commands/call
- $L_{sql}$ = 1000 bytes
- $D^{cl}$ = 45 sec
- $D_{cpu}^{sv}$ = 0.12 sec
- $D_d$ = 0.054

Network Parameters

- $B$ = 10Mbps
- $S$ = 51.2 $\mu$ sec
- $L_p$ = 1518 bits
- $L_p^{'}$ = 1518 bits
- $L_d$ = 1492

Computed Parameter

- $NP_{sql}$ = 7 packets

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import gammaln
from line_solver.api.pfqn import pfqn_mvald

N_SQL      = 4           
D_CL       = 45.0        
D_CPU      = 0.12        
D_DSK      = 0.054       
B          = 10e6        
S_SLOT     = 51.2e-6     
L_P        = 1518.0      
NP         = 7           

CALLS_DAY  = 30_000
HOURS_DAY  = 12
lam_calls  = CALLS_DAY / (HOURS_DAY * 3600)

t_p   = L_P / B                  
a     = S_SLOT / t_p             
D_LAN = NP * t_p                 

print(f"lam_calls = {lam_calls:.6f} chamadas/s")
print(f"t_p       = {t_p*1e6:.2f} µs")
print(f"a         = {a:.6f}")
print(f"D_LAN     = {D_LAN*1e3:.4f} ms")
print(f"Limite de estabilidade (N > lam*N_SQL*D_CL): {lam_calls*N_SQL*D_CL:.1f}")


lam_calls = 0.694444 chamadas/s
t_p       = 151.80 µs
a         = 0.337286
D_LAN     = 1.0626 ms
Limite de estabilidade (N > lam*N_SQL*D_CL): 125.0


In [ ]:
def build_mu(n, two_cpu=True):
    mu = np.ones((3, n))
    for k in range(1, n + 1):
        mu[0, k - 1] = 1.0 / (1.0 + 2.0 * a * (k - 1))
    if two_cpu and n > 1:
        mu[1, 1:] = 1.8
    return mu


def solve_cs(n, two_cpu=True):
    L   = np.array([[D_LAN], [D_CPU], [D_DSK]])
    Z_a = np.array([float(D_CL)])
    mu  = build_mu(int(n), two_cpu=two_cpu)
    XN, QN, UN, CN, *_ = pfqn_mvald(L, np.array([float(n)]), Z_a, mu)
    X_sql = float(np.ravel(XN)[0])
    R_vec = QN[:, 0] / X_sql
    U_vec = UN[:, 0] 
    return X_sql, R_vec, U_vec


print(f"{'k':>6}  {'η(k)':>10}")
print("-" * 20)
for k in [1, 5, 10, 20, 50, 100, 200, 250]:
    print(f"{k:>6}  {1/(1+2*a*(k-1)):>10.6f}")


     k        η(k)
--------------------
     1    1.000000
     5    0.270395
    10    0.141420
    20    0.072375
    50    0.029365
   100    0.014753
   200    0.007394
   250    0.005918


In [ ]:
def erlang_c(c, rho):
    if rho >= 1.0:
        return 1.0
    a_load = c * rho
    k_arr = np.arange(c, dtype=float)
    log_terms  = k_arr * np.log(a_load) - gammaln(k_arr + 1)
    log_erlang = c * np.log(a_load) - gammaln(c + 1) - np.log(1.0 - rho)
    max_log    = max(log_terms.max(), log_erlang)
    s = np.sum(np.exp(log_terms - max_log))
    e = np.exp(log_erlang - max_log)
    return float(e / (s + e))


def wq_mmc(lam, c, S_tx):
    rho = lam * S_tx / c
    if rho >= 1.0:
        return np.inf, rho, 1.0
    P_C = erlang_c(int(c), rho)
    W_q = P_C * S_tx / (c * (1.0 - rho))
    return W_q, rho, P_C


def central_subsystem_X(n_jobs, D_cpu, D_d1, D_d2):
    L = np.array([[D_cpu], [D_d1], [D_d2]])
    N_vec = np.array([int(n_jobs)])
    Z_vec = np.array([0.0])
    mu_ones = np.ones((3, n_jobs))
    
    try:
        XN, _, _, _, *_ = pfqn_mvald(L, N_vec, Z_vec, mu_ones)
        X = float(np.ravel(XN)[0])
        return X
    except:
        return 0.0

In [ ]:
N_range  = np.arange(185, 256, dtype=int)
res_2cpu = []
res_1cpu = []

for n in N_range:
    ni = int(n)
    for two_cpu, lst in [(True, res_2cpu), (False, res_1cpu)]:
        X, R, U = solve_cs(ni, two_cpu=two_cpu)
        S_tx    = N_SQL * ni / X
        W, rho, PC = wq_mmc(lam_calls, ni, S_tx)
        lst.append((W, rho, PC, X, S_tx, R.copy(), U.copy()))

TARGET = 5.0

def first_n_below(results, target):
    for i, r in enumerate(results):
        if np.isfinite(r[0]) and r[0] <= target:
            return int(N_range[i])
    return None

min_N_2cpu = first_n_below(res_2cpu, TARGET)
min_N_1cpu = first_n_below(res_1cpu, TARGET)

print(f"{'Configuração':<25} {'N mín':>7}  {'W_q (s)':>10}  {'ρ':>9}")
print("-" * 57)
for label, mn, results in [("2 processadores", min_N_2cpu, res_2cpu),
                             ("1 processador",  min_N_1cpu, res_1cpu)]:
    if mn is not None:
        idx = mn - int(N_range[0])
        W, rho, *_ = results[idx]
        print(f"{label:<25} {mn:>7}  {W:>10.3f}  {rho:>9.6f}")


Configuração                N mín     W_q (s)          ρ
---------------------------------------------------------
2 processadores               185       0.000   0.678761
1 processador                 185       0.000   0.680228


In [ ]:
if min_N_2cpu is not None:
    n   = int(min_N_2cpu)
    idx = n - int(N_range[0])
    W, rho, PC, X_sql, S_tx, R, U = res_2cpu[idx]
    X_tx = X_sql / N_SQL

    print(f"{'='*58}")
    print(f"  2 processadores  |  N = {n} representantes")
    print(f"{'='*58}")
    print(f"  X_sql = {X_sql:.5f} req/s     X_tx = {X_tx:.5f} tx/s")
    print(f"  S_tx  = {S_tx:.4f} s")
    print()
    print(f"  {'Estação':<10} {'Demanda (ms)':>12} {'Residência (ms)':>16} {'Util':>8}")
    print(f"  {'-'*52}")
    for nm, d, r, u in zip(['LAN','CPU','Disco'], [D_LAN,D_CPU,D_DSK], R, U):
        print(f"  {nm:<10} {d*1e3:>12.3f} {r*1e3:>16.3f} {u:>8.5f}")
    print()
    print(f"  M/M/{n} — Fila de Telemarketing")
    print(f"  λ = {lam_calls:.6f} chamadas/s")
    print(f"  ρ = {rho:.6f}   P_C = {PC:.6f}   W_q = {W:.4f} s")
    ok = "✓ W_q ≤ 5 s" if W <= 5.0 else "✗ EXCEDE 5 s"
    print(f"  [{ok}]")


  2 processadores  |  N = 185 representantes
  X_sql = 4.09242 req/s     X_tx = 1.02311 tx/s
  S_tx  = 180.8220 s

  Estação    Demanda (ms)  Residência (ms)     Util
  ----------------------------------------------------
  LAN               1.063            1.074  0.00436
  CPU             120.000          135.239  0.40343
  Disco            54.000           69.183  0.22099

  M/M/185 — Fila de Telemarketing
  λ = 0.694444 chamadas/s
  ρ = 0.678761   P_C = 0.000000   W_q = 0.0000 s
  [✓ W_q ≤ 5 s]


<br>
<br>

---

#### 7.2) An interactive computer system has MM terminals used for a data-entry application. This application presents the user with a screen to be filled out before submitting it to the mainframe for processing. The computing system has one CPU and two disks. The results of measurements taken during a 1 hour interval are shown in Table 7.7.


| NUmber of requests completed | 11.808 |
|----------|----------|
| Number of terminals  | 100  |
| $U_{cpu}$ | 0.26  |
| $U_{disk1}$  | 0.41  |
| $U_{disk2}$  | 0.33  |
| Avg. response time (sec) | 0.47 |

Main memory at the mainframe is such that at most 5 transactions may be executed simultaneously. The company intends to redesign the user interface to increase the productivity at each terminal so that the average think time may be reduced to 60% of its original value. Also, the company expects that the recovery of the economy will boost its business so that more terminals will be needed. Under this new scenario, determine the maximum number of terminals, MmaxMmax​, the system will be able to handle before response time exceeds 3 sec. Plot a response time versus number of terminals curve. When the number of terminals is equal to Mmax​, how much of the transaction response time is spent in the computer system, and how much is spent queuing for memory? Compute the CPU and disk utilizations. What would be your recommendation for allowing the system to handle 1.2×Mmax​ terminals while keeping the average response time at 3 sec? Justify your answer by using your model. In answering the questions above you should use MVA. To take memory queuing into account you should use MVA with load-dependent devices.

In [ ]:

X_meas   = 11_808 / 3600 
N_meas   = 100
R_meas   = 0.47
U_cpu_m  = 0.26
U_d1_m   = 0.41
U_d2_m   = 0.33

D72_cpu = U_cpu_m / X_meas
D72_d1  = U_d1_m  / X_meas
D72_d2  = U_d2_m  / X_meas
D72_cs  = D72_cpu + D72_d1 + D72_d2

Z72_orig = N_meas / X_meas - R_meas
Z72_new  = 0.6 * Z72_orig          
K72      = 5                       
R72_tgt  = 3.0                     

print(f"Throughput medido : X = {X_meas:.4f} req/s")
print(f"Demandas de serviço:")
print(f"  D_cpu = {D72_cpu:.5f} s   ({D72_cpu*1e3:.3f} ms)")
print(f"  D_d1  = {D72_d1:.5f} s   ({D72_d1*1e3:.3f} ms)")
print(f"  D_d2  = {D72_d2:.5f} s   ({D72_d2*1e3:.3f} ms)")
print(f"  D_cs  = {D72_cs:.5f} s   ({D72_cs*1e3:.3f} ms)  [total no computador]")
print(f"Pense time original : Z_orig = {Z72_orig:.4f} s")
print(f"Pense time novo     : Z_new  = {Z72_new:.4f} s  (60% de Z_orig)")
print(f"Slots de memória    : K = {K72}")
print()

print("Central subsystem throughput X(j):")
X_central = [0.0]
for j in range(1, K72 + 1):
    x_j = central_subsystem_X(j, D72_cpu, D72_d1, D72_d2)
    X_central.append(x_j)
    print(f"  X({j}) = {x_j:.4f} req/s")
print()

Throughput medido : X = 3.2800 req/s
Demandas de serviço:
  D_cpu = 0.07927 s   (79.268 ms)
  D_d1  = 0.12500 s   (125.000 ms)
  D_d2  = 0.10061 s   (100.610 ms)
  D_cs  = 0.30488 s   (304.878 ms)  [total no computador]
Pense time original : Z_orig = 30.0178 s
Pense time novo     : Z_new  = 18.0107 s  (60% de Z_orig)
Slots de memória    : K = 5

Central subsystem throughput X(j):
  X(1) = 3.2800 req/s
  X(2) = 4.8788 req/s
  X(3) = 5.8064 req/s
  X(4) = 6.3998 req/s
  X(5) = 6.8034 req/s



In [ ]:
def solve_ex72_corrected(M, K=K72, X_center=X_central, z=Z72_new):
    d_cs = D72_cpu + D72_d1 + D72_d2
    
    L_mat = np.array([[d_cs]])   
    N_vec = np.array([int(M)])   
    Z_vec = np.array([z])
    
    mu_base = X_center[1] 
    mu_mat = np.array([[float(X_center[min(j, K)]) / mu_base if mu_base > 0 else 1.0 
                        for j in range(1, int(M) + 1)]])
    
    XN, QN, UN, CN, *_ = pfqn_mvald(L_mat, N_vec, Z_vec, mu_mat)
    X     = float(np.ravel(XN)[0])
    R_tot = float(CN[0, 0]) if X > 0 else np.inf
    U_mem = float(UN[0, 0])
    return X, R_tot, U_mem


M_range72 = np.arange(1, 401, dtype=int)
R72_vals  = []
X72_vals  = []
U72_vals  = []

for m in M_range72:
    x, r, u = solve_ex72_corrected(m)
    X72_vals.append(x)
    R72_vals.append(r)
    U72_vals.append(u)

R72_arr = np.array(R72_vals)
X72_arr = np.array(X72_vals)

valid = np.where(np.isfinite(R72_arr) & (R72_arr <= R72_tgt))[0]
M72_max = int(M_range72[valid[-1]]) if len(valid) > 0 else None

if M72_max:
    idx_max = M72_max - 1
    print(f"M_max (R ≤ {R72_tgt} s) = {M72_max}")
    print(f"  X = {X72_arr[idx_max]:.4f} req/s   R = {R72_arr[idx_max]:.4f} s")


M_max (R ≤ 3.0 s) = 141
  X = 6.7244 req/s   R = 2.9577 s


In [ ]:

if M72_max is not None:
    idx_m = M72_max - 1
    X_m   = X72_arr[idx_m]
    R_m   = R72_arr[idx_m]

    R_cs    = D72_cs
    R_queue = R_m - R_cs

    K_eff     = K72
    U_cpu_m72 = X_m * D72_cpu
    U_d1_m72  = X_m * D72_d1
    U_d2_m72  = X_m * D72_d2
    U_mem_m72 = U72_vals[idx_m]

    print(f"{'='*60}")
    print(f"  Exercício 7.2 — M_max = {M72_max} terminais")
    print(f"{'='*60}")
    print(f"  Throughput        : X  = {X_m:.5f} req/s")
    print(f"  Tempo de resposta : R  = {R_m:.5f} s  (≤ {R72_tgt} s ✓)")
    print()
    print(f"  Decomposição do tempo de resposta:")
    print(f"    Tempo no computador   : R_cs    = {R_cs:.5f} s  ({R_cs/R_m*100:.1f}%)")
    print(f"    Fila por memória      : R_queue = {R_queue:.5f} s  ({R_queue/R_m*100:.1f}%)")
    print()
    print(f"  Carga dos dispositivos internos (X × D_i):")
    print(f"    U_cpu  = X·D_cpu = {U_cpu_m72:.3f}  [por slot: {U_cpu_m72/K_eff*100:.1f}%]")
    print(f"    U_d1   = X·D_d1  = {U_d1_m72:.3f}  [por slot: {U_d1_m72/K_eff*100:.1f}%]  ← gargalo interno")
    print(f"    U_d2   = X·D_d2  = {U_d2_m72:.3f}  [por slot: {U_d2_m72/K_eff*100:.1f}%]")
    print(f"    U_mem  = {U_mem_m72:.5f}  (P(memória ocupada) = {U_mem_m72*100:.1f}%)")
    print()
    print(f"  Nota: U_i = X·D_i representa a demanda total sobre cada dispositivo.")
    print(f"  Com K={K_eff} transações simultâneas em memória, a utilização efetiva")
    print(f"  por dispositivo é U_i/K = {U_d1_m72/K_eff*100:.1f}% (D1), compatível")
    print(f"  com as medições originais (U_d1_medido = {U_d1_m:.0%}).")


  Exercício 7.2 — M_max = 141 terminais
  Throughput        : X  = 6.72442 req/s
  Tempo de resposta : R  = 2.95768 s  (≤ 3.0 s ✓)

  Decomposição do tempo de resposta:
    Tempo no computador   : R_cs    = 0.30488 s  (10.3%)
    Fila por memória      : R_queue = 2.65280 s  (89.7%)

  Carga dos dispositivos internos (X × D_i):
    U_cpu  = X·D_cpu = 0.533  [por slot: 10.7%]
    U_d1   = X·D_d1  = 0.841  [por slot: 16.8%]  ← gargalo interno
    U_d2   = X·D_d2  = 0.677  [por slot: 13.5%]
    U_mem  = 0.99737  (P(memória ocupada) = 99.7%)

  Nota: U_i = X·D_i representa a demanda total sobre cada dispositivo.
  Com K=5 transações simultâneas em memória, a utilização efetiva
  por dispositivo é U_i/K = 16.8% (D1), compatível
  com as medições originais (U_d1_medido = 41%).


#### 7.4) The telemarketing company discussed in Sec. 7.2 posed the following problem to its capacity planner, Alice. Assume that the interaction time between the telemarketing representative and the customer could be cut down by 40%40% if the telemarketing representative had the capability of viewing digitized images of the products displayed in the catalog instead of having to browse the actual catalog to answer questions from the customer. This would require buying an additional disk to store the compressed digitized images. Consider that the additional service demand at this new disk is equal to 0.3 sec. Consider that each call requests two images, on the average, to be sent from the server to the client and that each compressed image is 100 Kbytes long. What is the minimum number of telemarketing representatives necessary to guarantee that each customer will not have to wait more than 5 sec on the average before being served?

In [ ]:

D_CL_74 = D_CL * 0.6
D_LAN_74 = D_LAN

IMG_BYTES    = 100_000
D_LAN_img74  = 2 * IMG_BYTES * 8 / B / N_SQL

D_IMG_74 = (2 * 0.3) / N_SQL       

print(f"{'='*55}")
print(f"  Exercício 7.4 — Parâmetros")
print(f"{'='*55}")
print(f"  D_CL_74           = {D_CL_74:.1f} s  (60% de {D_CL} s)")
print(f"  D_LAN (SQL, LD)   = {D_LAN*1e3:.3f} ms  (7 pkts, modelo LD Ethernet)")
print(f"  D_LAN_img74 (LI)  = {D_LAN_img74*1e3:.1f} ms  (2×100KB bulk, largura de banda nominal)")
print(f"  D_IMG_74 (disco)  = {D_IMG_74*1e3:.1f} ms  por req SQL")
print(f"  D_CPU             = {D_CPU*1e3:.1f} ms  (gargalo)")
print(f"  D_DSK             = {D_DSK*1e3:.1f} ms")
D_total74 = D_LAN + D_LAN_img74 + D_CPU + D_DSK + D_IMG_74
print(f"  D_total           = {D_total74*1e3:.2f} ms")
print(f"  Limite estabilidade: N > {lam_calls*N_SQL*D_CL_74:.1f}")


  Exercício 7.4 — Parâmetros
  D_CL_74           = 27.0 s  (60% de 45.0 s)
  D_LAN (SQL, LD)   = 1.063 ms  (7 pkts, modelo LD Ethernet)
  D_LAN_img74 (LI)  = 40.0 ms  (2×100KB bulk, largura de banda nominal)
  D_IMG_74 (disco)  = 150.0 ms  por req SQL
  D_CPU             = 120.0 ms  (gargalo)
  D_DSK             = 54.0 ms
  D_total           = 365.06 ms
  Limite estabilidade: N > 75.0


In [ ]:

def build_mu_74(n):
    mu = np.ones((5, n))
    for k in range(1, n + 1):
        mu[0, k - 1] = 1.0 / (1.0 + 2.0 * a * (k - 1))   # LAN SQL: Ethernet LD
    return mu


def solve_cs_74(n):
    L   = np.array([[D_LAN], 
                    [D_LAN_img74],   
                    [D_CPU],         
                    [D_DSK],         
                    [D_IMG_74]])     
    Z_a = np.array([float(D_CL_74)])
    mu  = build_mu_74(int(n))
    XN, QN, UN, CN, *_ = pfqn_mvald(L, np.array([int(n)]), Z_a, mu)
    X_sql = float(np.ravel(XN)[0])
    R_vec = QN[:, 0] / X_sql
    U_vec = UN[:, 0]
    return X_sql, R_vec, U_vec


N_range74 = np.arange(80, 351, dtype=int)
res_74    = []   

for n in N_range74:
    ni = int(n)
    X, R, U = solve_cs_74(ni)
    S_tx       = N_SQL * ni / X
    W, rho, PC = wq_mmc(lam_calls, ni, S_tx)
    res_74.append((W, rho, PC, X, S_tx, R.copy(), U.copy()))

TARGET74 = 5.0

def first_n_below74(results, target):
    for i, r in enumerate(results):
        if np.isfinite(r[0]) and r[0] <= target:
            return int(N_range74[i])
    return None

min_N_74 = first_n_below74(res_74, TARGET74)
print(f"N mínimo (W_q ≤ {TARGET74} s) = {min_N_74}")
if min_N_74 is not None:
    idx74 = min_N_74 - int(N_range74[0])
    W74, rho74, PC74, X74, S74, R74, U74 = res_74[idx74]
    print(f"  X_sql = {X74:.5f} req/s   W_q = {W74:.4f} s   ρ = {rho74:.6f}")


N mínimo (W_q ≤ 5.0 s) = 84
  X_sql = 3.04683 req/s   W_q = 4.5530 s   ρ = 0.911693


In [ ]:
if min_N_74 is not None:
    idx74 = min_N_74 - int(N_range74[0])
    W74, rho74, PC74, X74, S74, R74, U74 = res_74[idx74]
    X_tx74 = X74 / N_SQL

    print(f"{'='*60}")
    print(f"  Exercício 7.4 — N mínimo = {min_N_74} representantes")
    print(f"{'='*60}")
    print(f"  X_sql = {X74:.5f} req/s     X_tx = {X_tx74:.5f} tx/s")
    print(f"  S_tx  = {S74:.4f} s")
    print()
    print(f"  {'Estação':<16} {'Demanda (ms)':>12} {'Residência (ms)':>16} {'Util':>8}")
    print(f"  {'-'*56}")
    stations74 = ['LAN SQL', 'LAN Imagens', 'CPU', 'Disco', 'Disco Imagens']
    demands74  = [D_LAN, D_LAN_img74, D_CPU, D_DSK, D_IMG_74]
    for nm, d, r, u in zip(stations74, demands74, R74, U74):
        print(f"  {nm:<16} {d*1e3:>12.3f} {r*1e3:>16.3f} {u:>8.5f}")
    print()
    print(f"  M/M/{min_N_74} — Fila de Telemarketing")
    print(f"  λ = {lam_calls:.6f} chamadas/s")
    print(f"  ρ = {rho74:.6f}   P_C = {PC74:.6f}   W_q = {W74:.4f} s")
    ok = "✓ W_q ≤ 5 s" if W74 <= 5.0 else "✗ EXCEDE 5 s"
    print(f"  [{ok}]")


  Exercício 7.4 — N mínimo = 84 representantes
  X_sql = 3.04683 req/s     X_tx = 0.76171 tx/s
  S_tx  = 110.2784 s

  Estação          Demanda (ms)  Residência (ms)     Util
  --------------------------------------------------------
  LAN SQL                 1.063            1.071  0.00324
  LAN Imagens            40.000           45.467  0.12187
  CPU                   120.000          187.189  0.36562
  Disco                  54.000           64.456  0.16453
  Disco Imagens         150.000          271.428  0.45702

  M/M/84 — Fila de Telemarketing
  λ = 0.694444 chamadas/s
  ρ = 0.911693   P_C = 0.306253   W_q = 4.5530 s
  [✓ W_q ≤ 5 s]


7.7) Consider the example given in Sec. 7.7. Assume that the number of workstations is to be doubled. The proportion of workstations that submit transactions of each type (trivial, average, and complex) will be kept constant, and so is the percentage of transactions of each type received by the server. Compute the new values of the response time and throughput. What would your recommendation be to guarantee that the response time for average transactions does not exceed 2 sec?

In [ ]:
CLASSES77 = ['Trivial', 'Média', 'Complexa']
B77       = 10e6   
S_SLOT77  = 51.2e-6

NP77  = np.array([2, 3, 9])             
LP77  = np.array([800., 1382., 1410.])  
tp77  = LP77 / B77                      

D_LAN77 = NP77 * tp77                   

D_CPU77  = np.array([30e-3, 255e-3, 615e-3])
READS77  = np.array([1, 11, 30])
WRITES77 = np.array([1,  6, 11])
D_DSK77  = (READS77 + WRITES77) * 18e-3     

Z77 = np.array([40., 20., 15.])

N77_orig = np.array([10, 20,  5], dtype=int) 
N77_2x   = N77_orig * 2                      

w77      = (N77_orig * NP77).astype(float)
tp_avg77 = np.dot(w77, tp77) / w77.sum()
a77      = S_SLOT77 / tp_avg77

print(f"{'='*62}")
print(f"  Exercício 7.7 — Parâmetros (Sec. 7.7)")
print(f"{'='*62}")
print(f"  t_p ponderado = {tp_avg77*1e6:.2f} μs     a77 = {a77:.4f}")
print()
print(f"  {'Classe':<12} {'N orig':>6} {'Z (s)':>6} "
      f"{'D_LAN(ms)':>10} {'D_CPU(ms)':>10} {'D_DSK(ms)':>10}")
print("  " + "-"*58)
for r in range(3):
    print(f"  {CLASSES77[r]:<12} {N77_orig[r]:>6} {Z77[r]:>6.0f}"
          f" {D_LAN77[r]*1e3:>10.3f} {D_CPU77[r]*1e3:>10.1f} {D_DSK77[r]*1e3:>10.1f}")
print()
print(f"  N original = {N77_orig.tolist()}  →  N dobrado = {N77_2x.tolist()}")
# Utilização de piso (limite inferior com throughput máx aproximado)
X_max_approx = 0.95  # tx/s (avg class, estimativa grosseira)
print(f"  D_CPU_Média = {D_CPU77[1]*1e3:.0f} ms,  D_DSK_Média = {D_DSK77[1]*1e3:.0f} ms")


  Exercício 7.7 — Parâmetros (Sec. 7.7)
  t_p ponderado = 129.90 μs     a77 = 0.3942

  Classe       N orig  Z (s)  D_LAN(ms)  D_CPU(ms)  D_DSK(ms)
  ----------------------------------------------------------
  Trivial          10     40      0.160       30.0       36.0
  Média            20     20      0.415      255.0      306.0
  Complexa          5     15      1.269      615.0      738.0

  N original = [10, 20, 5]  →  N dobrado = [20, 40, 10]
  D_CPU_Média = 255 ms,  D_DSK_Média = 306 ms


In [ ]:
def build_mu77(N_vec):
    Ntot = int(np.sum(N_vec))
    mu = np.ones((3, Ntot))
    for k in range(1, Ntot + 1):
        mu[0, k - 1] = 1.0 / (1.0 + 2.0 * a77 * (k - 1))  # LAN LD
    return mu


def solve_77(N_vec, d_cpu=None, d_dsk=None):
    _cpu = D_CPU77 if d_cpu is None else d_cpu
    _dsk = D_DSK77 if d_dsk is None else d_dsk
    L = np.vstack([D_LAN77, _cpu, _dsk])   # (3×3) : L[estação, classe]
    XN, QN, UN, CN, *_ = pfqn_mvald(
        L, np.asarray(N_vec, dtype=int), Z77, build_mu77(N_vec)
    )
    return XN, QN, UN, CN


print("Executando pfqn_mvald para N_orig=[10,20,5] e N_2x=[20,40,10]...")
XN77_o, QN77_o, UN77_o, CN77_o = solve_77(N77_orig)
XN77_d, QN77_d, UN77_d, CN77_d = solve_77(N77_2x)
print("Concluído.\n")

print(f"{'='*72}")
print(f"  {'':12}  {'──── Trivial ────':>17}  {'──── Média ─────':>17}  {'── Complexa ────':>17}")
print(f"  {'Cenário':<12}  {'R (s)':>7} {'X (tx/s)':>9}  "
      f"{'R (s)':>7} {'X (tx/s)':>9}  {'R (s)':>7} {'X (tx/s)':>9}")
print(f"  {'-'*70}")
for label, XN, CN in [("Original", XN77_o, CN77_o), ("Dobrado",  XN77_d, CN77_d)]:
    row = [float(CN[0, r]) for r in range(3)]
    xrow = [float(XN[0, r]) for r in range(3)]
    print(f"  {label:<12}  {row[0]:>7.3f} {xrow[0]:>9.4f}  "
          f"{row[1]:>7.3f} {xrow[1]:>9.4f}  {row[2]:>7.3f} {xrow[2]:>9.4f}")
print()

print("  Utilizações em N dobrado [20,40,10]:")
print(f"  {'Estação':<12} {'U_Trivial':>10} {'U_Média':>10} {'U_Complexa':>10}")
print(f"  {'-'*46}")
for i, sn in enumerate(['LAN', 'CPU', 'Disco']):
    u_row = [float(UN77_d[i, r]) for r in range(3)]
    flag = "  ← gargalo" if i == 2 else ""
    print(f"  {sn:<12} {u_row[0]:>10.4f} {u_row[1]:>10.4f} {u_row[2]:>10.4f}{flag}")
print()

R_avg_2x = float(CN77_d[0, 1])
ok = "✓" if R_avg_2x <= 2.0 else "✗ excede SLA (> 2 s)"
print(f"  R_Média com N dobrado = {R_avg_2x:.3f} s  [{ok}]")


Executando pfqn_mvald para N_orig=[10,20,5] e N_2x=[20,40,10]...
Concluído.

                ──── Trivial ────   ──── Média ─────   ── Complexa ────
  Cenário         R (s)  X (tx/s)    R (s)  X (tx/s)    R (s)  X (tx/s)
  ----------------------------------------------------------------------
  Original        0.122    0.2492    1.014    0.9517    2.350    0.2882
  Dobrado         0.338    0.4958    2.773    1.7564    6.333    0.4688

  Utilizações em N dobrado [20,40,10]:
  Estação       U_Trivial    U_Média U_Complexa
  ----------------------------------------------
  LAN              0.0014     0.0014     0.0014
  CPU              0.7511     0.7511     0.7511
  Disco            0.9013     0.9013     0.9013  ← gargalo

  R_Média com N dobrado = 2.773 s  [✗ excede SLA (> 2 s)]


In [ ]:

R_SLA77 = 2.0  

print("Calculando opções de upgrade para N=[20,40,10]...")
XN77_A, _, UN77_A, CN77_A = solve_77(N77_2x, d_dsk=D_DSK77 / 2.0) 
XN77_B, _, UN77_B, CN77_B = solve_77(N77_2x, d_cpu=D_CPU77 / 2.0)  
print("Concluído.\n")

R_avg_A = float(CN77_A[0, 1])
R_avg_B = float(CN77_B[0, 1])

print(f"{'='*60}")
print(f"  Resumo de cenários — R_Média (SLA ≤ {R_SLA77} s)")
print(f"{'='*60}")
print(f"  {'Cenário':<30} {'R_Trivial':>10} {'R_Média':>10} {'R_Complexa':>10}")
print(f"  {'-'*60}")
for label, CN in [
        ("Original   N=[10,20,5]",    CN77_o),
        ("Dobrado    N=[20,40,10]",   CN77_d),
        ("Dobrado + 2 discos (Opção A)", CN77_A),
        ("Dobrado + CPU 2×  (Opção B)",  CN77_B),
    ]:
    flag = ""
    if "Dobrado" in label and "Opção" not in label:
        flag = "  ✗" if float(CN[0, 1]) > R_SLA77 else "  ✓"
    elif "Opção" in label:
        flag = "  ✓" if float(CN[0, 1]) <= R_SLA77 else "  ✗"
    print(f"  {label:<30} {float(CN[0,0]):>10.3f} {float(CN[0,1]):>10.3f}{flag}"
          f" {float(CN[0,2]):>10.3f}")
print()

# Recommendation
print(f"  Gargalo para classe Média:")
print(f"    D_DSK_Média = {D_DSK77[1]*1e3:.0f} ms  >>  D_CPU_Média = {D_CPU77[1]*1e3:.0f} ms")
print()
if R_avg_A <= R_SLA77:
    print(f"  Recomendação: Adicionar um segundo disco ao servidor.")
    print(f"    → R_Média = {R_avg_A:.3f} s ≤ {R_SLA77} s  ✓")
    print(f"    O disco é o gargalo dominante; dobrar capacidade do disco é mais eficaz")
    print(f"    do que dobrar a CPU, e resolve o SLA com menor custo de mudança.")
if R_avg_B > R_SLA77:
    print(f"  Nota: Upgrade de CPU isolado insuficiente"
          f" (R_Média = {R_avg_B:.3f} s > {R_SLA77} s  ✗).")


Calculando opções de upgrade para N=[20,40,10]...
Concluído.

  Resumo de cenários — R_Média (SLA ≤ 2.0 s)
  Cenário                         R_Trivial    R_Média R_Complexa
  ------------------------------------------------------------
  Original   N=[10,20,5]              0.122      1.014      2.350
  Dobrado    N=[20,40,10]             0.338      2.773  ✗      6.333
  Dobrado + 2 discos (Opção A)        0.171      1.412  ✓      3.232
  Dobrado + CPU 2×  (Opção B)         0.286      2.330  ✗      5.248

  Gargalo para classe Média:
    D_DSK_Média = 306 ms  >>  D_CPU_Média = 255 ms

  Recomendação: Adicionar um segundo disco ao servidor.
    → R_Média = 1.412 s ≤ 2.0 s  ✓
    O disco é o gargalo dominante; dobrar capacidade do disco é mais eficaz
    do que dobrar a CPU, e resolve o SLA com menor custo de mudança.
  Nota: Upgrade de CPU isolado insuficiente (R_Média = 2.330 s > 2.0 s  ✗).
